# Shared data preparation

Run these cells once at the top of each training notebook. They rebuild missing manifests instead of stopping.

In [ ]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
from collections import Counter
from datetime import datetime

RUNPOD_ROOT = Path("/workspace/SKN27-FINAL-3Team")
PROJECT_ROOT = None
if RUNPOD_ROOT.exists() and (RUNPOD_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = RUNPOD_ROOT
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "ai").exists() and (candidate / "storage").exists():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("project root not found")
MANIFEST_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/manifests"
RAW_VIDEO_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/raw_videos"
FRAME_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/frames"
SAMPLE_MANIFEST = MANIFEST_DIR / "sample_700_coarse_manifest.csv"
DOWNLOAD_MANIFEST = MANIFEST_DIR / "train_700_download_manifest.csv"
FRAME_MANIFEST = MANIFEST_DIR / "frame_manifest_train_700_f8.csv"

PER_LABEL = 700
FRAMES_PER_VIDEO = 8
IMAGE_SIZE = 224
SEED = 42
DEVICE = "auto"
NUM_WORKERS = 4

print("PROJECT_ROOT:", PROJECT_ROOT)
print("python:", sys.executable)


## Command helper

In [ ]:
def run_command(command, *, timeout=None):
    command = [str(part) for part in command]
    print("\n$", " ".join(command), flush=True)
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=timeout)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed


## Environment and disk check

In [ ]:
usage = shutil.disk_usage(PROJECT_ROOT)
print("total_gb:", round(usage.total / 1024**3, 2))
print("used_gb:", round(usage.used / 1024**3, 2))
print("free_gb:", round(usage.free / 1024**3, 2))
run_command([sys.executable, "-c", "import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"])


## Check sample manifest

In [ ]:
if not SAMPLE_MANIFEST.exists():
    raise FileNotFoundError(f"sample manifest not found: {SAMPLE_MANIFEST}")
with SAMPLE_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    sample_rows = list(csv.DictReader(f))
print("sample_rows:", len(sample_rows))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in sample_rows)))
print("split_counts:", dict(Counter(row.get("split") for row in sample_rows)))


## Build or check download manifest

In [ ]:
if not DOWNLOAD_MANIFEST.exists():
    run_command([
        sys.executable,
        "etl/vision/download_sampled_media.py",
        "--input", SAMPLE_MANIFEST,
        "--output", DOWNLOAD_MANIFEST,
        "--download-dir", RAW_VIDEO_DIR,
        "--label-column", "coarse_label",
        "--per-label", str(PER_LABEL),
        "--split", "",
    ])

with DOWNLOAD_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    download_rows = list(csv.DictReader(f))
print("download_rows:", len(download_rows))
print("download_status_counts:", dict(Counter(row.get("download_status") for row in download_rows)))
print("file_exists_counts:", dict(Counter(row.get("file_exists") for row in download_rows)))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in download_rows)))


# ResNet18 frame-level training

This notebook trains only the ResNet18 frame classifier. It runs all listed experiments without skip flags.

## Build or check frame manifest

In [ ]:
run_command([
    sys.executable,
    "etl/vision/extract_training_frames.py",
    "--input", DOWNLOAD_MANIFEST,
    "--output", FRAME_MANIFEST,
    "--frame-dir", FRAME_DIR,
    "--label-column", "coarse_label",
    "--frames-per-video", str(FRAMES_PER_VIDEO),
    "--overwrite",
])

with FRAME_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    frame_rows = list(csv.DictReader(f))
print("frame_rows:", len(frame_rows))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in frame_rows)))
print("split_counts:", dict(Counter(row.get("split") for row in frame_rows)))


## ResNet18 shared experiment helpers

In [ ]:
MODEL_DIR = PROJECT_ROOT / "storage/vision/models/classification"
EARLY_STOPPING_PATIENCE = 3
PRETRAINED = True
DETERMINISTIC = True


In [ ]:
def build_resnet_command(experiment):
    command = [
        sys.executable,
        "ai/vision/train_classifier.py",
        "--manifest", FRAME_MANIFEST,
        "--root-dir", PROJECT_ROOT,
        "--output-dir", MODEL_DIR,
        "--label-column", "coarse_label",
        "--model-name", "resnet18",
        "--image-size", str(IMAGE_SIZE),
        "--epochs", str(experiment["epochs"]),
        "--batch-size", str(experiment["batch_size"]),
        "--learning-rate", str(experiment["learning_rate"]),
        "--weight-decay", str(experiment["weight_decay"]),
        "--label-smoothing", str(experiment["label_smoothing"]),
        "--early-stopping-patience", str(EARLY_STOPPING_PATIENCE),
        "--seed", str(SEED),
        "--device", DEVICE,
        "--num-workers", str(NUM_WORKERS),
    ]
    if PRETRAINED:
        command.append("--pretrained")
    if experiment["freeze_backbone"]:
        command.append("--freeze-backbone")
    if DETERMINISTIC:
        command.append("--deterministic")
    else:
        command.append("--no-deterministic")
    return command


def latest_run_dir(output_dir):
    runs = [path for path in output_dir.iterdir() if path.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No run directories found: {output_dir}")
    return max(runs, key=lambda path: path.stat().st_mtime)


def show_run_result(run_dir):
    config_path = run_dir / "run_config.json"
    history_path = run_dir / "training_history.csv"
    model_path = run_dir / "model.pt"
    print("run_dir:", run_dir)
    print("model_exists:", model_path.exists(), model_path)

    if config_path.exists():
        config = json.loads(config_path.read_text(encoding="utf-8"))
        print("run_config")
        for key in [
            "run_id", "model_name", "pretrained", "freeze_backbone", "epochs",
            "batch_size", "learning_rate", "weight_decay", "label_smoothing",
            "early_stopping_patience", "best_epoch", "best_val_accuracy",
            "image_size", "seed", "device", "train_rows", "val_rows", "test_rows",
        ]:
            if key in config:
                print(f"- {key}: {config[key]}")

    if history_path.exists():
        rows = list(csv.DictReader(history_path.open("r", encoding="utf-8")))
        print("training_history")
        for row in rows:
            print(row)
        if rows:
            best_val = max(rows, key=lambda row: float(row.get("val_accuracy") or 0))
            best_test = max(rows, key=lambda row: float(row.get("test_accuracy") or 0))
            print("best_val:", best_val)
            print("best_test:", best_test)


## Combination 1 - freeze baseline

Define this experiment.

In [ ]:
EXPERIMENT = {'name': 'resnet_baseline_freeze_lr1e-3_e5', 'epochs': 5, 'batch_size': 32, 'learning_rate': 0.001, 'weight_decay': 0.0, 'label_smoothing': 0.0, 'freeze_backbone': True}
print(EXPERIMENT)


## Combination 1 - freeze baseline training

Run this experiment.

In [ ]:
run_command(build_resnet_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 1 - freeze baseline result

Review this experiment result.

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 2 - unfreeze lr 1e-4

Define this experiment.

In [ ]:
EXPERIMENT = {'name': 'resnet_exp2_unfreeze_lr1e-4_e10', 'epochs': 10, 'batch_size': 32, 'learning_rate': 0.0001, 'weight_decay': 0.0, 'label_smoothing': 0.0, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 2 - unfreeze lr 1e-4 training

Run this experiment.

In [ ]:
run_command(build_resnet_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 2 - unfreeze lr 1e-4 result

Review this experiment result.

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 3 - unfreeze lr 3e-5

Define this experiment.

In [ ]:
EXPERIMENT = {'name': 'resnet_exp3_unfreeze_lr3e-5_e10', 'epochs': 10, 'batch_size': 32, 'learning_rate': 3e-05, 'weight_decay': 0.0, 'label_smoothing': 0.0, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 3 - unfreeze lr 3e-5 training

Run this experiment.

In [ ]:
run_command(build_resnet_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 3 - unfreeze lr 3e-5 result

Review this experiment result.

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 4 - unfreeze lr 1e-5

Define this experiment.

In [ ]:
EXPERIMENT = {'name': 'resnet_exp4_unfreeze_lr1e-5_e15', 'epochs': 15, 'batch_size': 32, 'learning_rate': 1e-05, 'weight_decay': 0.0, 'label_smoothing': 0.0, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 4 - unfreeze lr 1e-5 training

Run this experiment.

In [ ]:
run_command(build_resnet_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 4 - unfreeze lr 1e-5 result

Review this experiment result.

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 5 - regularized lr 3e-5

Define this experiment.

In [ ]:
EXPERIMENT = {'name': 'resnet_exp5_unfreeze_lr3e-5_wd5e-2_ls0.05_e10', 'epochs': 10, 'batch_size': 32, 'learning_rate': 3e-05, 'weight_decay': 0.05, 'label_smoothing': 0.05, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 5 - regularized lr 3e-5 training

Run this experiment.

In [ ]:
run_command(build_resnet_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 5 - regularized lr 3e-5 result

Review this experiment result.

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 6 - long run lr 1e-4 wd 1e-2

Define this experiment.

In [ ]:
EXPERIMENT = {'name': 'resnet_exp6_unfreeze_lr1e-4_wd1e-2_e30', 'epochs': 30, 'batch_size': 32, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'label_smoothing': 0.0, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 6 - long run lr 1e-4 wd 1e-2 training

Run this experiment.

In [ ]:
run_command(build_resnet_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 6 - long run lr 1e-4 wd 1e-2 result

Review this experiment result.

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 7 - long run regularized lr 1e-4

Define this experiment.

In [ ]:
EXPERIMENT = {'name': 'resnet_exp7_unfreeze_lr1e-4_wd5e-2_ls0.05_e30', 'epochs': 30, 'batch_size': 32, 'learning_rate': 0.0001, 'weight_decay': 0.05, 'label_smoothing': 0.05, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 7 - long run regularized lr 1e-4 training

Run this experiment.

In [ ]:
run_command(build_resnet_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 7 - long run regularized lr 1e-4 result

Review this experiment result.

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 8 - long run regularized lr 5e-5

Define this experiment.

In [ ]:
EXPERIMENT = {'name': 'resnet_exp8_unfreeze_lr5e-5_wd5e-2_ls0.05_e30', 'epochs': 30, 'batch_size': 32, 'learning_rate': 5e-05, 'weight_decay': 0.05, 'label_smoothing': 0.05, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 8 - long run regularized lr 5e-5 training

Run this experiment.

In [ ]:
run_command(build_resnet_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 8 - long run regularized lr 5e-5 result

Review this experiment result.

In [ ]:
show_run_result(LAST_RUN_DIR)
